# **Parallel Chains** 

Parallel chains mein hum multiple operations(Task || component) ko simultaneously execute kar sakte hain.

# RunnableParallel
LangChain Core provides RunnableParallel to execute multiple Runnable components concurrently.

In [ ]:
# Ex : Suppose user gives a topic: Python Decorators.

# We want:
# - Task 1: Generate a short summary.
# - Task 2: Generate important keywords.
# Both tasks will execute simultaneously.




from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

llm = init_chat_model("gpt-5-mini",temperature = 0)

# Common prompt for both tasks
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful Python teacher."),
    ("human", "Explain {topic} in simple words.")
])

# Common chain
common_chain = prompt | llm | StrOutputParser()

# Task 1: Generate summary
summary_prompt = ChatPromptTemplate.from_messages([
    ("human", "Give a short summary of {topic}.")
])

summary_chain = summary_prompt | llm | StrOutputParser()


# Task 2: Generate keywords
keyword_prompt = ChatPromptTemplate.from_messages([
    ("human", "Give 5 important keywords related to {topic}.")
])

keyword_chain = keyword_prompt | llm | StrOutputParser()


# Run both chains in parallel
parallel_chain = RunnableParallel(
    summary=summary_chain,
    keywords=keyword_chain
)      # runnbale parrallel object create kar rahe 

#Python mein brackets ke andar expression ko multiple lines mein break kar sakte hain.
# Connect both stages
chain = (
    common_chain
    | (lambda text: {"text": text}) #string to dictionary
    | parallel_chain
)

# Execute both tasks simultaneously
response = chain.invoke({
    "topic": "Python Decorators"
})


print(response)


# Conditional Chains 

Abb agar hame conmdtional lagni hoo ki input yes /no (yeh kuch aise hoo jaise uppercase hoo) hoo tabhi task execute hoo to hum RunnableConditional ka use karenge.

In [ ]:
#So we use RunnableBranch
from langchain_core.runnables import RunnableBranch, RunnableLambda

def is_python_question(data):
    return "python" in data["topic"].lower()

def is_java_question(data):
    return "java" in data["topic"].lower()

python_chain = RunnableLambda(
    lambda data: f"Explaining Python: {data['topic']}"
)

java_chain = RunnableLambda(
    lambda data: f"Explaining Java: {data['topic']}"
)

default_chain = RunnableLambda(
    lambda data: "Sorry, I can explain only Python or Java."
)

branch = RunnableBranch(
    (is_python_question, python_chain),
    (is_java_question, java_chain),
    default_chain
)
#Iska matlab:
# 1. Pehle is_python_question check karo. True hua toh python_chain chalao.
# 2. False hua toh is_java_question check karo. True hua toh java_chain chalao.
# 3. Dono false hue toh default_chain chalao.

result = branch.invoke({"topic": "Python decorators"})

print(result)


# Important points:
# - Conditions order mein check hoti hain.
# - Jo pehli condition True hoti hai, uski chain execute hoti hai.
# - Koi condition match nahi hui toh default chain execute hoti hai.
# - RunnableBranch automatic AI reasoning nahi karta; conditions hum define karte hain.
# RunnableBranch vs LangGraph Conditional Edges:
# - RunnableBranch: LCEL chain ke andar conditional routing.
# - Conditional Edges: LangGraph mein ek node ke baad decide karna ki next node kaunsa execute hoga.